# Setup

Local fetch

In [ ]:
import gc
import torch
import pandas as pd
import os

# Load data
augment_url = "diversity_sampling/dataset/coresets/augment.csv"
augment = pd.read_csv(augment_url)[['index', 'review', 'sentiment']]

DB fetch alternative

In [ ]:
import gc
import torch
import pandas as pd
import os
from diversity_sampling.database.api import get_augment_set

# Load data
augment = pd.DataFrame(await get_augment_set)

In [ ]:
from diversity_sampling.models.augmentation import AugmentModel
# Initialize model
augmenter = AugmentModel()

os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

***

Due to limited resource, augmentation process divided into 2 phase
* 1. Generate & Store data
* 2. Sampling using diversity-oriented selection & add to synthetic-set

# Loop

In [ ]:
start = 0

for i in range(1000):
    # Loop config
    if i != 0: 
        start += 10
    end = start + 10
    
    print(f"Augmenting indexes: {start} - {end}")
    temp = pd.DataFrame(augment).iloc[[x for x in range(start, end)]]
    
    # Main runner
    candidates = augmenter.augment(dataset=temp, n_candidates=5)
    torch.save(candidates, f"diversity_sampling/storage/augment_chunks/augment_chunk_{start}_{end}.pt")
    print(f"Milestone saved. Total objects stored: {len(candidates)}")
        
    # Clear memory & re-allocate memory for consistency
    del candidates, temp
    gc.collect()
    torch.cuda.synchronize()
    torch.cuda.empty_cache()

In [ ]:
from diversity_sampling.models import Candidates

torch.serialization.add_safe_globals([Candidates])

In [ ]:
# Detach generation, embedding model to make space for label-classification model
import gc
model = augmenter.model.to("cpu")
del model
gc.collect()
embedding_model = augmenter.embedding_model.to("cpu")
del embedding_model
gc.collect()
torch.cuda.empty_cache()

# Load classification model
augmenter.label_classifier = augmenter._load_label_classifier(augmenter.model_id)

In [ ]:
# Load augmented data
chunk_holder = []
start = 0
stop = 10000
for i in range((stop-start) // 10):
    if i != 0:
        start += 10
    end = start + 10
    chunk_holder.extend(torch.load(f"diversity_sampling/storage/augment_chunks/augment_chunk_{start}_{end}.pt"))

In [ ]:
# Diversity-oriented selection
result = augmenter.diversity_measurement(chunk_holder)

In [ ]:
# Clear memory
clf = augmenter.label_classifier.to("cpu")
del clf
gc.collect()
torch.cuda.empty_cache()

# Local Storage

In [ ]:
selected_data = pd.concat([pd.DataFrame(), pd.DataFrame({'review': result['selected'], 'sentiment': result['labels']})], axis=0)

selected_storage = "diversity_sampling/dataset/augment_sets/high_quality.csv"

pd.read_csv(selected_storage)

In [ ]:
# Selected
pd.concat([pd.read_csv(selected_storage), selected_data], axis=0).reset_index(drop=True)['review', 'sentiment'].to_csv(selected_storage, index=False)

# Cloud storage

In [ ]:
import pandas as pd 
from diversity_sampling.database.api import insert_table

selected_storage = "diversity_sampling/dataset/augment_sets/high_quality.csv"

high_quality_set = pd.read_csv(selected_storage)

insert_table("high_quality", high_quality_set, "augment_sets")
